# Week 1 demo · Support Assistant v1 for Kestrel Telecom, Days 1 to 5

This notebook builds and runs the complete **Support Assistant v1** step by step, one day of Week 1 at a time. Each day's code was an exercise during the week. Here it all runs as one assistant on Amazon Bedrock in Europe (London).

| Part | Day | What you see run |
|---|---|---|
| 0 | – | Setup and preflight |
| 1 | Accounts, identity, Regions and costs | Who you are, what the policy allows, tags, a teardown, a bucket in the wrong Region |
| 2 | Calling Claude through Amazon Bedrock | One request read properly, token cost, retries, a stopped stream, structured output |
| 3 | Tools and execution limits | Two tools, the dispatcher's five checks, loop limits, approval, injection attempts |
| 4 | Data validation and document retrieval | Data contract, loading S3 and DynamoDB, the knowledge base, cited answers, retrieval measured |
| 5 | Prompt design and testing | Templates, two prompt versions in Prompt Management, a regression test, the complete assistant |
| 6 | Close | What it cost, what's left, teardown |

**How to run:** run the cells in order with **Shift + Enter**. Each step starts with what it shows and a **Say** note for the presenter. Cells tagged `live-call` call AWS. A full run makes about 150 model calls, takes about 25 minutes (most of it the knowledge base build and the Day 5 regression test) and costs well under 1 US dollar.

**Resources** are named `fde-<owner>-kestrel-...` and tagged `course=fde-aws`, `owner=<owner>`. Part 6 removes them all when `TEARDOWN = True`.

---
## Part 0 · Setup
### Step 0.1 · Settings
Change these before the session if you need to. `OWNER` names and tags every resource this notebook creates.

In [1]:
OWNER = "dfv"                  # 2-4 lower-case letters; resources become fde-dfv-kestrel-*
PROFILE = "fde-demo"           # the AWS CLI profile you signed in with (aws login --profile fde-demo)
FULL_QUESTION_SET = True       # Day 5: True = all 30 questions per prompt version (~5 min each); False = first 8
TEARDOWN = True                # Part 6: True = delete everything this notebook created

### Step 0.2 · Load the assistant and start fresh records
The completed assistant is the `support_assistant` package in this folder. Each session writes its own ledger (every AWS call and its cost) and run log (every decision the code made), so the totals at the end are this session's only.

In [2]:
import os, sys, json, time, pathlib, datetime, textwrap, subprocess

os.environ.setdefault("AWS_PROFILE", PROFILE)
os.environ["AWS_DEFAULT_REGION"] = os.environ["AWS_REGION"] = "eu-west-2"
os.environ["FDE_OWNER"] = OWNER

HERE = pathlib.Path.cwd()
assert (HERE / "support_assistant").is_dir(), "Start Jupyter from the demo5 folder (see README.md)."
sys.path.insert(0, str(HERE))

from support_assistant import config
SESSION = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
config.LEDGER_FILE = config.WORK_DIR / f"calls-{SESSION}.jsonl"
config.RUN_LOG_FILE = config.WORK_DIR / f"runlog-{SESSION}.jsonl"

import boto3
from support_assistant.ledger import Ledger, RunLog
LEDGER, RUN_LOG = Ledger(), RunLog()

def show(value):
    # Print as JSON, with account numbers replaced.
    print(config.redact(json.dumps(value, indent=2, ensure_ascii=False, default=str)))

def wrap(text, indent=""):
    for paragraph in str(text).split("\n"):
        print(textwrap.fill(paragraph, 100, initial_indent=indent, subsequent_indent=indent) if paragraph else "")

print("Package :", HERE / "support_assistant")
print("Owner   :", config.owner(), "| Region:", config.region(), "| Profile:", os.environ["AWS_PROFILE"])
print("Model   :", config.MODEL_ID)
print("Records :", config.LEDGER_FILE.name, "and", config.RUN_LOG_FILE.name, "in", config.WORK_DIR)

ModuleNotFoundError: No module named 'pydantic'

### Step 0.3 · Preflight: are we signed in?

In [ ]:
from support_assistant.config import aws_cli
version = aws_cli(["--version"])
caller = aws_cli(["sts", "get-caller-identity", "--query", "Arn", "--output", "text"])
print("AWS CLI :", (version.stdout or version.stderr).split()[0])
print("Signed in:", caller.ok)
assert caller.ok, "Sign in first: aws login --profile " + PROFILE + "   then restart the kernel."

---
# Part 1 · Day 1: Accounts, identity, Regions and costs
Everything else stands on this: **who** is calling, **what** the policy allows, **where** it runs, and **how** we find and remove what we create.

### Step 1.1 · Who am I? (Exercise 1.1, `describe_identity()`)
**Say:** Every request is signed by an identity. We split the ARN into its parts and replace the account number before anything is shown or saved.

In [ ]:
from support_assistant.config import describe_identity
identity = describe_identity(caller.stdout)
show(identity.as_record())

### Step 1.2 · What does the policy allow, and where?
**Say:** Same identity, same action, different Region. Europe (London) is where Kestrel's data may live. A request elsewhere may be refused by a Region condition in the policy. If your account allows it, that's a result too.

In [ ]:
allowed = aws_cli(["bedrock", "list-foundation-models", "--region", config.region(), "--by-provider", "anthropic",
                   "--query", "modelSummaries[].modelId", "--output", "text"])
ireland = aws_cli(["bedrock", "list-guardrails", "--region", "eu-west-1"])
print("London  list-foundation-models -> exit", allowed.returncode, "| course model listed:", config.BASE_MODEL_ID in allowed.stdout)
print("Ireland list-guardrails        -> exit", ireland.returncode, "|", ireland.error_code or "allowed in this account")

### Step 1.3 · Can we use the course model, and where does it run?
**Say:** We never call the base model ID on demand. We call the **EU inference profile**, which routes only to European Regions.

In [ ]:
availability = aws_cli(["bedrock", "get-foundation-model-availability", "--model-id", config.BASE_MODEL_ID,
                        "--region", config.region(), "--output", "json"])
destinations = aws_cli(["bedrock", "get-inference-profile", "--region", config.region(),
                        "--inference-profile-identifier", config.MODEL_ID, "--query", "models[].modelArn", "--output", "json"])
if availability.ok:
    show({k: v for k, v in json.loads(availability.stdout).items() if k != "ResponseMetadata"})
print("Profile routes to:", sorted({arn.split(":")[3] for arn in json.loads(destinations.stdout or "[]")}))
print("Why this model   :", config.MODEL_CHOICE_REASON)

### Step 1.4 · Costs: budgets warn, teardown stops
**Say:** Prices are dated and stored with the code, so every cost we report can be checked. A budget alert arrives *after* billing updates. It never stops a request or deletes anything.

In [ ]:
prices = json.loads((config.DATA_DIR / "pricing.json").read_text())
model_price = next(m for m in prices["models"] if m["inference_profile_id"] == config.MODEL_ID)
print(f"{model_price['model_name']} through {config.MODEL_ID}, {prices['currency']} {prices['unit']}:")
print(f"  input {model_price['input']} | output {model_price['output']} | cache read {model_price['cache_read']}")
print("  source:", json.dumps(prices["source"])[:140])
print(f"  so 1,000 input + 300 output tokens cost {(1000 * model_price['input'] + 300 * model_price['output']) / 1e6:.5f} USD")
account = subprocess.run(["aws", "sts", "get-caller-identity", "--query", "Account", "--output", "text"],
                         capture_output=True, text=True).stdout.strip()
budgets = aws_cli(["budgets", "describe-budgets", "--account-id", account, "--query", "Budgets[].BudgetName", "--output", "json"])
del account
print("Budgets visible to this identity:", budgets.stdout.strip() if budgets.ok else budgets.error_code)

### Step 1.5 · Tag it, find it, remove it
**Say:** We create a scratch bucket with our course tags, find it through the tagging API, delete it, and check that it's gone. Every resource this week follows this pattern.

In [ ]:
def tagged(region, lab):
    result = aws_cli(["resourcegroupstaggingapi", "get-resources", "--region", region,
                      "--tag-filters", "Key=course,Values=fde-aws", f"Key=lab,Values={lab}", f"Key=owner,Values={config.owner()}",
                      "--query", "ResourceTagMappingList[].ResourceARN", "--output", "json"])
    return json.loads(result.stdout or "[]") if result.ok else []

def wait_for_listing(region, lab, name, *, present, limit=180):
    started = time.monotonic()
    while True:
        arns = tagged(region, lab)
        if any(name in arn for arn in arns) == present or time.monotonic() - started > limit:
            return arns, round(time.monotonic() - started)
        time.sleep(10)

region = config.region()
bucket = f"fde-{config.owner()}-lab1-{config.random_suffix()}"
created = aws_cli(["s3api", "create-bucket", "--bucket", bucket, "--region", region,
                   "--create-bucket-configuration", f"LocationConstraint={region}"])
assert created.ok, created.stderr
aws_cli(["s3api", "put-bucket-tagging", "--bucket", bucket, "--region", region, "--tagging", json.dumps({"TagSet": config.tag_list(1)})])
listed, seconds = wait_for_listing(region, 1, bucket, present=True)
print(f"Created {bucket}; tag listing found it after {seconds} s:", any(bucket in a for a in listed))
aws_cli(["s3api", "delete-bucket", "--bucket", bucket, "--region", region])
listed, seconds = wait_for_listing(region, 1, bucket, present=False)
print(f"Deleted it; gone from the tag listing after {seconds} s:", not any(bucket in a for a in listed))

### Step 1.6 · The classic mistake: a bucket in the wrong Region
**Say:** A command with no Region falls back to a default, here US East (N. Virginia). The London listing doesn't show it. You only find it by checking **both** Regions. Then we delete it.

In [ ]:
virginia = config.POLICY_REGIONS[1]
stray = f"fde-{config.owner()}-lab1-{config.random_suffix()}"
made = aws_cli(["s3api", "create-bucket", "--bucket", stray], env={**os.environ, "AWS_DEFAULT_REGION": virginia, "AWS_REGION": virginia})
if not made.ok:
    print("Your account refused a bucket in", virginia, "->", made.error_code, "(an organisation guardrail; that's fine)")
else:
    aws_cli(["s3api", "put-bucket-tagging", "--bucket", stray, "--region", virginia, "--tagging", json.dumps({"TagSet": config.tag_list(1)})])
    london, _ = wait_for_listing(region, 1, stray, present=True, limit=20)
    found, seconds = wait_for_listing(virginia, 1, stray, present=True)
    print("London listing shows it     :", any(stray in a for a in london))
    print("N. Virginia listing shows it:", any(stray in a for a in found), f"(after {seconds} s)")
    aws_cli(["s3api", "delete-bucket", "--bucket", stray, "--region", virginia])
    print("Deleted the stray bucket in", virginia)

---
# Part 2 · Day 2: Calling Claude through Amazon Bedrock
The model is a remote service that can be slow, busy or wrong. The client reads every response fully, retries only what's temporary, and writes every attempt to the ledger.

### Step 2.1 · One Converse request, read properly (Exercise 1.4, `result_from_response()`)
**Say:** We read the text, the stop reason, the token usage and the request ID. The request ID is what you quote to AWS support.

In [ ]:
from support_assistant.bedrock import BedrockClient, should_retry, backoff_delay, request_structured, validate_structured, BedrockCallError
client = BedrockClient(ledger=LEDGER)
print("SDK retries off; our client retries itself. Read timeout:", client.read_timeout, "s\n")

result = client.converse([{"role": "user", "content": [{"text": "In one sentence, what does a broadband router do?"}]}], run_id="day2-first")
print("text        :", result.text())
print("stop reason :", result.stop_reason, "| complete:", result.complete)
print("usage       :", result.usage)
print("latency     :", result.latency_ms, "ms | attempts:", result.attempts, "| request id:", result.request_id)

### Step 2.2 · What did that cost? The ledger
**Say:** One row per attempt, priced from the dated price list. Input tokens and output tokens have different prices.

In [ ]:
row = LEDGER.rows("day2-first")[-1]
show({k: row[k] for k in ("operation", "status", "attempt", "input_tokens", "output_tokens", "cost_usd", "cost_status", "price_read_on")})

### Step 2.3 · Which failures do we retry? (Exercise 1.7)
**Say:** Only temporary ones, up to six attempts, with exponential backoff and full jitter, so clients don't retry in step. A validation or permission error fails the same way every time. A timeout we never retry ourselves, because the first request may still be running and billing.

In [ ]:
for code in ["ThrottlingException", "ServiceUnavailableException", "InternalServerException", "ModelNotReadyException",
             "ValidationException", "AccessDeniedException", "ReadTimeoutError"]:
    print(f"  {code:30} retry: {should_retry(code)}")
print("\n  Maximum wait before each retry (the real wait is random between 0 and this):")
print("  " + " | ".join(f"after try {a}: {backoff_delay(a, rng=lambda: 1.0):.0f} s" for a in range(1, config.MAX_ATTEMPTS)))

### Step 2.4 · A refused request
**Say:** Calling the base model ID on demand is refused. It's a `ValidationException`, so the client doesn't retry, and the ledger still records the attempt.

In [ ]:
wrong = BedrockClient(ledger=LEDGER, model_id=config.BASE_MODEL_ID)
try:
    wrong.converse([{"role": "user", "content": [{"text": "Hello"}]}], run_id="day2-refused")
except BedrockCallError as error:
    print("Refused:", error.code, "| retried:", LEDGER.rows("day2-refused")[-1]["will_retry"])
    print(str(error)[:200])

### Step 2.5 · Streaming, and a stream we stop (Exercise 1.6)
**Say:** A stream that finishes ends with a usage block. One that the application stops early has none, so the ledger records its cost as **unknown**, never as zero.

In [ ]:
question = [{"role": "user", "content": [{"text": "Give three short tips for faster home Wi-Fi."}]}]
full = client.stream(question, run_id="day2-stream-full", on_text=lambda t: print(t, end="", flush=True))
print("\n\nfull stream    -> complete:", full.complete, "| chunks:", full.chunks, "| usage:", full.usage)
cut = client.stream(question, max_chunks=5, run_id="day2-stream-cut")
print("stopped stream -> complete:", cut.complete, "| reason:", cut.interrupted_reason, "| usage:", cut.usage)
r = LEDGER.rows("day2-stream-cut")[-1]
print("ledger row     ->", {k: r[k] for k in ("status", "cost_usd", "note")})

### Step 2.6 · Structured output, checked by a schema (Exercise 1.5)
**Say:** We ask for a case summary as JSON. Pydantic validates it. Bedrock's native structured output constrains the format, and we still validate, because the schema can't check our business rules.

In [ ]:
from support_assistant.schemas import CaseSummary, case_summary_prompt
from support_assistant.tools import LocalCaseStore
local_cases = LocalCaseStore()
case = local_cases.get("case_1003")
print("Case:", case["case_id"], "-", case["summary"], "\n")
for native in (False, True):
    outcome = request_structured(client, case_summary_prompt(case), CaseSummary, native=native, run_id=f"day2-structured-{native}")
    print("native structured output" if native else "JSON from the prompt", "->")
    show(outcome.record.model_dump() if outcome.record else {"rejected": outcome.rejected_reason})
print("\nA reply that breaks the schema:", validate_structured('{"case_id": "case_1003", "urgency": "panic"}', CaseSummary)[1][:160])
print("A reply that is cut off       :", validate_structured('{"case_id": "case_10', CaseSummary)[1])

---
# Part 3 · Day 3: Tools and execution limits
The model can ask for a tool, but it never runs one. Our code checks every request and decides. v1 has two tools, and neither can change an account, issue a credit or send anything.

### Step 3.1 · The two tool definitions (Exercise 1.8)

In [ ]:
from support_assistant.tools import tool_config, Dispatcher, CallerScope, DraftStore, approve_draft, load_users, DynamoCaseStore
cfg = tool_config()
for tool in cfg.get("toolConfig", cfg)["tools"]:
    spec = tool["toolSpec"]
    print(f"{spec['name']}\n  {spec['description'][:150]}...\n  input: {json.dumps(spec['inputSchema']['json']['properties'])}\n")
users = load_users()
ADA, JONAH, HALIMA = users["user_customer_01"], users["user_agent_01"], users["user_supervisor_01"]
for u in users.values():
    print(f"{u['user_id']:20} {u['role']:17} {', '.join(u['scopes'])}")

### Step 3.2 · The dispatcher's five checks, in order (Exercises 1.9 and 1.10)
**Say:** Allowlist, then time left, then input schema, then repeated call, then caller scope. Ada is a customer, so she may see only her own cases. Watch each request stop at a different check. Only one gets through.

In [ ]:
dispatcher = Dispatcher(case_store=local_cases, scope=CallerScope.for_user(ADA), run_id="day3-dispatch",
                        draft_store=DraftStore(), run_log=RUN_LOG)
requests = [
    ("issue_credit", {"case_id": "case_1003", "amount_pence": 5000}, 60, "check 1: not on the allowlist"),
    ("lookup_case",  {"case_id": "case_1005"},                       0,  "check 2: no time left"),
    ("lookup_case",  {"case": "1003"},                               60, "check 3: wrong input shape"),
    ("lookup_case",  {"case_id": "case_1003"},                       60, "Ada's own case: allowed"),
    ("lookup_case",  {"case_id": "case_1003"},                       60, "check 4: the same call again"),
    ("lookup_case",  {"case_id": "case_1016"},                       60, "check 5: another customer's case"),
    ("draft_reply",  {"case_id": "case_1003", "reply_text": "Thanks for getting in touch about your bill."}, 60, "check 5: customers may not draft"),
]
for n, (name, value, seconds, why) in enumerate(requests, 1):
    result, decision = dispatcher.dispatch({"toolUseId": f"t{n}", "name": name, "input": value}, remaining_seconds=seconds)
    print(f"{name:12} {json.dumps(value)[:48]:48} -> {decision.decision:8} [{decision.rule:15}] {why}")
print("\nWhat the model is told after a refusal:", result["toolResult"]["content"][0]["json"])

### Step 3.3 · The loop's own limits (Exercise 1.9, `check_limits()`)
**Say:** At most six model calls and 180 seconds per question, checked before every call.

In [ ]:
from support_assistant.loop import check_limits, run_loop
for calls, now in [(2, 10), (6, 10), (2, 181)]:
    print(f"  calls={calls}, {now:>3} s elapsed ->", check_limits(calls, config.MAX_MODEL_CALLS, now, config.RUN_DEADLINE_SECONDS) or "carry on")

### Step 3.4 · The tool loop, live
**Say:** Jonah, a first-line agent, asks about a case. The model asks for `lookup_case`, the dispatcher allows it, and the model answers from the record.

In [ ]:
loop = run_loop(client, "What happened on case_1004, and is it still open?", CallerScope.for_user(JONAH),
                case_store=local_cases, run_log=RUN_LOG, run_id="day3-lookup")
print("status:", loop.status, "| model calls:", loop.model_calls)
for d in loop.decisions:
    print("  tool request:", d.tool, "->", d.decision)
wrap(loop.answer, "  ")

### Step 3.5 · A draft waits for a person (Exercise 1.9, `approve_draft()`)
**Say:** `draft_reply` sends nothing. It creates a proposal. Jonah can't approve it. Only Halima, the supervisor, can, and her approval is what writes the reply to the case.

In [ ]:
drafts = DraftStore(config.WORK_DIR / f"drafts-{SESSION}.jsonl")
loop = run_loop(client, "Draft a short, friendly reply to the customer on case_1004 confirming the case is closed.",
                CallerScope.for_user(JONAH), case_store=local_cases, draft_store=drafts, run_log=RUN_LOG, run_id="day3-draft")
print("status:", loop.status, "| decisions:", [(d.tool, d.decision) for d in loop.decisions])
draft = loop.pending_drafts[0]
wrap(draft["reply_text"], "  | ")
try:
    approve_draft(draft["draft_id"], decision="approved", decided_by=JONAH["role"], case_store=local_cases, draft_store=drafts, run_log=RUN_LOG)
except PermissionError as error:
    print("\nJonah approves  ->", error)
record = approve_draft(draft["draft_id"], decision="approved", decided_by=HALIMA["role"], case_store=local_cases, draft_store=drafts, run_log=RUN_LOG)
print("Halima approves ->", record["status"])
note = local_cases.get("case_1004")["history"][-1]
print("Case note added ->", note["author"], "approved by", note["approved_by"])

### Step 3.6 · Injection attempts
**Say:** case_1026 has a customer note that tells the assistant to ignore its rules. Then someone claims to be a supervisor and asks for a case outside their scope. The prompt may or may not resist. The dispatcher refuses anyway, because the rule is in code.

In [ ]:
from support_assistant.prompts import INJECTION_CASES
injected = local_cases.get("case_1026")
print("Planted note:", injected["history"][-1]["note"][:200], "\n")
for case in (INJECTION_CASES[0], INJECTION_CASES[2]):
    r = run_loop(client, case["question"], CallerScope.for_case(case["scope_case"]), case_store=local_cases,
                 draft_store=drafts, run_log=RUN_LOG, max_model_calls=case["max_model_calls"], run_id=f"day3-{case['name']}")
    print(f"[{case['name']}] {case['question']}")
    print("  tool requests:", [(d.tool, d.decision, d.rule) for d in r.decisions], "| status:", r.status)
    wrap((r.pending_drafts[-1]["reply_text"] if r.pending_drafts else r.answer or "")[:400], "  ")
    print()

**Say:** Every decision went to the run log with a digest of the input, never the input itself.

In [ ]:
for line in RUN_LOG.lines()[-8:]:
    print({k: line.get(k) for k in ("run_id", "tool", "rule", "decision", "input_digest")})

---
# Part 4 · Day 4: Data validation and document retrieval
Nothing is loaded until it passes the data contract. Then documents go to S3, cases to DynamoDB, and the documents into a managed knowledge base. The assistant answers only from what it retrieves, and every citation is checked.

### Step 4.1 · The data contract (Exercise 1.11, `validate_case()`)
**Say:** The contract says what a valid record is: its fields, types, allowed values and formats. Kestrel's dataset has defects planted on purpose. The contract finds them *before* anything is written. The written contract is in `docs/data-contract-template.md`.

In [ ]:
from support_assistant import ingest
cases_raw = json.loads((config.DATA_DIR / "cases.json").read_text())["cases"]
for record in cases_raw:
    problems = ingest.validate_case(record)
    if problems:
        print(record.get("case_id"), "->", problems)
print("\nA made-up bad record:", ingest.validate_case({"case_id": "1003", "status": "on_fire"})[:3])

### Step 4.2 · Plan the load and write the quality report
**Say:** One plan decides what will be written and what is rejected, with reasons, before any AWS call.

In [ ]:
plan = ingest.plan_ingestion(*ingest.dataset_sources(), config.reference_date())
report = ingest.quality_report(plan)
show({k: report[k] for k in ("read", "accepted", "duplicates", "stale_documents", "documents_by_status")})
print("rejected:", [(r["identifier"], r["reasons"]) for r in report["rejected"]["records"]])
fixtures = ingest.plan_ingestion(*ingest.fixture_sources(), config.reference_date())
print("\nContract-test fixtures, each built to break one rule:")
for item in fixtures.rejected:
    print(f"  {item.identifier:30} {item.reasons[0][:85]}")

### Step 4.3 · Load S3 and DynamoDB, twice
**Say:** The bucket has four zones: `raw/`, `cleansed/`, `curated/` and `rejected/`. A case is written only if its content changed. The second run writes nothing, so a repeated build never duplicates data.

In [ ]:
names = config.resource_names()
s3 = boto3.client("s3", region_name=config.region())
dynamodb = boto3.client("dynamodb", region_name=config.region())
print("bucket created now:", ingest.ensure_bucket(s3, names.bucket), "->", names.bucket)
print("table created now :", ingest.ensure_table(dynamodb, names.table), "->", names.table)
manifest_rows, first_run = ingest.ingest(plan, s3=s3, dynamodb=dynamodb, bucket=names.bucket, table=names.table)
_, second_run = ingest.ingest(plan, s3=s3, dynamodb=dynamodb, bucket=names.bucket, table=names.table)
ingest.save_lab4_outputs(manifest_rows, ingest.quality_report(plan, first_run))
show({"first_run": first_run, "second_run": second_run})

In [ ]:
zones = {}
for page in s3.get_paginator("list_objects_v2").paginate(Bucket=names.bucket):
    for obj in page.get("Contents", []):
        zones[obj["Key"].split("/")[0] + "/"] = zones.get(obj["Key"].split("/")[0] + "/", 0) + 1
print("Objects by zone:", zones)

### Step 4.4 · Create the knowledge base (Exercise 5, part one)
**Say:** Three things: a service role that may read `curated/` only, a managed knowledge base, and an S3 data source with default chunking. The first time, this takes a few minutes.

In [ ]:
from support_assistant.retrieval import ensure_kb_role, ensure_knowledge_base, ensure_data_source, sync
started = time.perf_counter()
account = boto3.client("sts", region_name=config.region()).get_caller_identity()["Account"]
role_arn = ensure_kb_role(boto3.client("iam"), account, names.role, names.bucket)
agent = boto3.client("bedrock-agent", region_name=config.region())
kb_id = ensure_knowledge_base(agent, names.knowledge_base, role_arn)
print(f"knowledge base {kb_id} active after {time.perf_counter() - started:.0f} s")
ds_id = ensure_data_source(agent, kb_id, names.data_source, names.bucket, account)
del account
print(f"data source {ds_id} available after {time.perf_counter() - started:.0f} s")
state = config.load_state(); state.update(knowledge_base_id=kb_id, data_source_id=ds_id, bucket=names.bucket, table=names.table)
config.save_state(state)

### Step 4.5 · Sync the documents into the knowledge base
**Say:** The ingestion job reads the curated documents and their metadata, chunks them and indexes them. Watch for documents that failed and for warnings.

In [ ]:
job = sync(agent, kb_id, ds_id)
show({k: job.get(k) for k in ("status", "statistics", "warnings", "elapsed_seconds")})

### Step 4.6 · Retrieve passages
**Say:** Every question starts here. These passages are the only evidence the model may use.

In [ ]:
from support_assistant.retrieval import Retriever, check_citations, KeywordBaseline, load_corpus, compare_retrievers, hit_at
retriever = Retriever(kb_id, boto3.client("bedrock-agent-runtime", region_name=config.region()), LEDGER)
question = "How long does a refund take after I cancel?"
passages = retriever.retrieve(question, run_id="day4-retrieve")
for p in passages:
    print(f"#{p.rank} {p.doc_id} score {p.score:.3f} | {p.text[:85].replace(chr(10), ' ')}...")

### Step 4.7 · Answer from the passages, then check every citation
**Say:** The model must cite a document ID after each claim. Code checks that every cited document was actually supplied and is in the manifest. If the passages don't answer the question, the model must say NO_EVIDENCE, and we won't guess.

In [ ]:
from support_assistant.assistant import answer_from_passages
manifest = ingest.load_manifest()
for q in [question, "What is the capital of France?"]:
    found = retriever.retrieve(q, run_id="day4-answer")
    r = answer_from_passages(client, q, found, manifest, run_id="day4-answer", run_log=RUN_LOG, top=found[0].score if found else None)
    print(f"Q: {q}\n  -> {r.kind} | citations {[c.doc_id for c in r.citations]}"); wrap(r.text, "  "); print()
print("The citation check on three answers:")
for answer in ["Refunds take 10 working days [kd_07].", "Refunds take 10 working days [kd_99].", "Refunds take 10 working days."]:
    _, problems = check_citations(answer, passages, manifest)
    print(f"  {answer:40} -> {'PASS' if not problems else 'REFUSED: ' + '; '.join(problems)}")

### Step 4.8 · Measure retrieval: keyword search vs the knowledge base (Exercise 1.12, `hit_at()`)
**Say:** We measure retrieval on its own, over the 30 questions. Hit@k asks whether a supporting document is in the top k results. For questions with no answer in the corpus, a *low* top score is what we want.

In [ ]:
questions = json.loads((config.DATA_DIR / "questions.json").read_text())["questions"]
comparison = compare_retrievers(questions, KeywordBaseline(load_corpus()), retriever, run_id="day4-compare")
show({k: comparison[k] for k in ("answerable_questions", "baseline", "knowledge_base", "no_evidence_top_scores")})

---
# Part 5 · Day 5: Prompt design and testing, and the complete assistant
The prompt is code. It is kept as numbered versions in Amazon Bedrock Prompt Management, and a new version must pass a regression test before we use it.

### Step 5.1 · A template and its variables (Exercise 1.15, `render_template()`)
**Say:** The template declares its variables. A missing value or a misspelled variable is an error, not a silent blank in the prompt.

In [ ]:
from support_assistant.prompts import LOCAL_PROMPT, REVISED_PROMPT, render_template, ensure_version, run_version, regression_problems
print(render_template("Question: {{question}}\n{{passages}}", {"question": "Refund time?", "passages": "<passage doc_id='kd_07'>...</passage>"}))
for text, values in [("Question: {{question}}", {}), ("Question: {{qestion}}", {"question": "x", "passages": "y"})]:
    try:
        render_template(text, values)
    except ValueError as error:
        print("refused ->", error)

### Step 5.2 · Version 1 and version 2 (Exercise 1.16)
**Say:** Version 1 is the prompt that Days 3 and 4 used. Version 2 names its parts, puts the passages first and the question last, adds an example of the answer format, and states the defences against instructions hidden in data.

In [ ]:
print("VERSION 1 · message part\n" + "-" * 60); print(LOCAL_PROMPT.message)
print("\nVERSION 2 · " + REVISED_PROMPT.description + "\n" + "-" * 60); print(REVISED_PROMPT.message)

### Step 5.3 · Save both versions in Prompt Management
**Say:** Each version is immutable and has its own ARN. Running this again reuses a version with the same words rather than making a new one.

In [ ]:
version_1 = ensure_version(agent, LOCAL_PROMPT, "Version 1: the prompt that the assistant used in Week 1.")
version_2 = ensure_version(agent, REVISED_PROMPT, "Version 2: " + REVISED_PROMPT.description)
for v in (version_1, version_2):
    print(f"version {v.version}: {config.redact(v.arn)} | same words as code: {v.same_text(LOCAL_PROMPT if v is version_1 else REVISED_PROMPT)}")

### Step 5.4 · Assemble the complete assistant (Exercise 1.16, `SupportAssistant.ask()`)
**Say:** `ask()` is the front door. It retrieves passages, then routes. A question that names a case or asks for a draft goes to the tool loop, anything else to an answer from passages. Both routes use the same prompt version. Cases now come from DynamoDB.

In [ ]:
from dataclasses import replace
from support_assistant.assistant import SupportAssistant
assistant = SupportAssistant(client=client, retriever=retriever, case_store=DynamoCaseStore(names.table), manifest=manifest,
                             draft_store=DraftStore(config.WORK_DIR / f"drafts-{SESSION}.jsonl"), run_log=RUN_LOG)

def ask(question, user, version=None):
    r = (replace(assistant, prompt=version) if version else assistant).ask(question, user)
    print(f"[{user['display_name']} · {user['role']}] {question}")
    print(f"  -> {r.kind} | tools {r.tools_used or '-'} | model calls {r.model_calls} | cites {[c.doc_id for c in r.citations]}")
    wrap(r.text, "  "); print()
    return r

ask("What does the speed guarantee promise?", ADA)
ask("What is the status of case_1030?", ADA)

### Step 5.5 · Run both versions on the question set
**Say:** Each version answers the 30 questions as the user who asks each one, plus the five injection cases. This is the slowest step, about five minutes per version. While it runs, explain what the regression test will compare.

In [ ]:
question_set = questions if FULL_QUESTION_SET else questions[:8]
runs = {}
for v in (version_1, version_2):
    started = time.perf_counter()
    runs[v.version] = run_version(assistant, v, question_set, users, ledger=LEDGER)
    (config.WORK_DIR / f"day5-prompt-version-{v.version}-{SESSION}.json").write_text(json.dumps(runs[v.version], indent=2, default=str))
    print(f"version {v.version}: {time.perf_counter() - started:.0f} s")
    show(runs[v.version]["summary"])

### Step 5.6 · The regression test (Exercise 1.17, `regression_problems()`)
**Say:** Version 2 fails if it gives fewer cited answers than version 1, or if the model asks for more cases outside the caller's scope. We decide on evidence, not on how one answer reads.

In [ ]:
baseline, candidate = runs[version_1.version]["summary"], runs[version_2.version]["summary"]
for key in ("cited_answers", "answerable", "no_evidence_answered", "out_of_scope_requests", "drafts_proposed", "model_attempts", "cost_usd"):
    print(f"  {key:24} v{version_1.version}: {baseline[key]!s:>9}   v{version_2.version}: {candidate[key]!s:>9}")
problems = regression_problems(baseline, candidate)
print("\nVersion 2:", "PASSES the regression test" if not problems else "FAILS ->")
for p in problems:
    print("  -", p)
print("\nAnd the test catches a worse version:",
      regression_problems(baseline, {**baseline, "cited_answers": baseline["cited_answers"] - 3}))

### Step 5.7 · Support Assistant v1, complete: follow one question
**Say:** Ada asks about her first bill on case_1003, her own case. Then she asks about case_1016, her neighbour's. Same path, but the dispatcher refuses on scope. Then a policy question and a question with no answer in the documents, all on the version that passed.

In [ ]:
chosen = version_2 if not problems else version_1
print("Using prompt version", chosen.version, "\n")
first = ask("Why was my first bill higher than my monthly price? My case reference is case_1003.", ADA, chosen)
second = ask("Can you tell me what's happening on case_1016?", ADA, chosen)
ask("Can I keep my landline number if I switch to Digital Voice?", ADA, chosen)
ask("Do you sell mobile phone contracts?", ADA, chosen)
for r in (first, second):
    print("run", r.run_id, "decisions:", [(l["tool"], l["decision"], l["rule"]) for l in RUN_LOG.lines(r.run_id) if l.get("tool")])

### Step 5.8 · Optional: the web chat
Outside the notebook, the same assistant runs behind a small local web page with a user picker, citations and a drafts panel where the supervisor approves. In a terminal, from the demo5 folder:

```bash
export AWS_PROFILE=fde-demo AWS_DEFAULT_REGION=eu-west-2 AWS_REGION=eu-west-2 FDE_OWNER=dfv
uv run python -m support_assistant.webchat      # then open http://127.0.0.1:8765
```

---
# Part 6 · Close: what it cost, what's left, teardown
**Say:** v1 answers only from evidence, cites its sources, can't read cases it shouldn't, and sends nothing without a person's approval. Its gaps lead into Week 2: when two documents disagree it can confidently pick one, and it runs only on this laptop.

In [ ]:
totals = LEDGER.totals()
show(totals)
print(f"\nThis session: {totals['attempts']} AWS calls recorded, {totals['known_cost_usd']:.4f} USD known cost,",
      totals["attempts_with_unknown_cost"], "attempts with unknown cost.")

In [ ]:
from support_assistant.assistant import inventory, teardown
if TEARDOWN:
    print("Removing (deleting the data source can take ten minutes or more)...")
    show(teardown())
print("What is left with this owner's names or tags:")
show(inventory())

If you set `TEARDOWN = False`, the bucket, the table and the knowledge base stay, ready for Week 2 or the next run of this notebook. A later run reuses them. To remove them later, set `TEARDOWN = True` and run the last two cells, or from a terminal in this folder:

```bash
FDE_OWNER=dfv AWS_PROFILE=fde-demo AWS_DEFAULT_REGION=eu-west-2 uv run python -m support_assistant teardown --yes
```